# VecDB Index & Model Operations
Navigate Oracle VecDB index lifecycle tasks, reusable model catalog exploration, annotation hygiene in a single guided notebook.


## 1. Scenario Overview
This guided workflow demonstrates:
1. Bootstrapping a dedicated demo table using shared `.env` credentials.
2. Inspecting baseline index metadata, then creating and validating an HNSW index with job logs.
3. Surfacing hosted models, capturing descriptive metadata, and documenting model governance fields.
4. Evolving table annotations for governance and dropping resources for repeatable runs.


## 2. Setup
Install the required VecDB SDK dependencies and authenticate once so every subsequent operation reuses the same configured `OracleVecDB` client.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas


In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification (self-signed certificates).')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for this workflow.')


## 3. Create Demo Table
Provision a fresh dense-vector table with descriptive annotations and seed deterministic sample documents that will fuel the remaining index and model workflows.


In [ ]:
from uuid import uuid4
from random import Random

print('Creating demo table and seeding data...')
TABLE = os.getenv('INDEX_MODEL_TABLE', 'INDEX_MODEL_DEMO')
vecdb.create_vector_table(
    name=TABLE,
    comment='Index/model demo',
    annotations={'TITLE': 'string', 'CATEGORY': 'string'},
)
print('Created table', TABLE)

def make_vec(seed, dim=10):
    rng = Random(seed)
    return [rng.random() for _ in range(dim)]

docs = [
    {'title': 'Predictive maintenance plan', 'category': 'analytics'},
    {'title': 'Customer onboarding workflow', 'category': 'cs'},
    {'title': 'Global marketing brief', 'category': 'marketing'},
    {'title': 'Risk mitigation summary', 'category': 'finance'},
]

rows = [
    {
        'id': str(uuid4()),
        'dense_vector': make_vec(i),
        'metadata': {'TITLE': doc['title'], 'CATEGORY': doc['category']}
    }
    for i, doc in enumerate(docs)
]
vecdb.upsert_vectors(table_name=TABLE, vectors=rows)
print('Seeded', len(rows), 'rows')


## 4. Describe Table
Query the table definition to capture baseline index metadata before introducing explicit HNSW structures.


In [ ]:
print('Describing table to inspect index metadata...')
vecdb.describe_vector_table(name=TABLE)


## 5. Create & Manage Index
Request a named HNSW index, then capture the asynchronous job roster to trace build progress and gather diagnostics.


In [ ]:
print('Creating HNSW index via index_params...')
index_params = {
    'vector_index_params': {
        'organization': 'INMEMORY GRAPH',
        'distance_metric': 'COSINE',
        'advanced_params': {
            'neighbors': 32,
            'efConstruction': 200
        },
    },
}
job = vecdb.create_index(table_name=TABLE, index_params=index_params)
print('Index job submitted:', job)

In [ ]:
print('Inspecting index details...')
print(vecdb.describe_index(table_name=TABLE))

print('Latest index job details...')
jobs = vecdb.list_index_jobs()
if jobs.items:
    latest_job = jobs.items[0]

    job_name = getattr(latest_job, 'job_name', None)

    if job_name:
        print(vecdb.describe_index_job(index_job_name=job_name))
        try:
            log = vecdb.get_index_job_log(index_job_name=job_name)
            log_repr = log.to_str() if hasattr(log, 'to_str') else str(log)
            print('Index job log snippet:', log_repr[:200])
        except Exception as exc:
            print('Unable to fetch index job log:', exc)
    else:
        print('Job GUID not available in response.')
else:
    print('No index jobs available')


## 6. Model Operations
Enumerate available VecDB models, dive into metadata for the first entry, and confirm the environment's catalog baseline.


In [ ]:
from pprint import pprint

print('Listing available models...')
models = vecdb.list_models()

if models.items:
    for item in models.items:
        summary = f"- {item.model_name} | algo={item.algorithm} | function={item.mining_function} | created={item.creation_date}"
        print(summary)
        for attr in item.attributes or []:
            print(f"    • {attr.name}: {attr.value} ({attr.data_type})")
else:
    print('No models available')

if models.items:
    name = models.items[0].model_name
    print('Describe model:', name)
    details = vecdb.describe_model(model_name=name)
    if hasattr(details, 'to_dict'):
        pprint(details.to_dict())
    else:
        pprint(details)


## 7. Update Table Annotations
Adjust the table's annotation schema to include an ownership field, demonstrating governance tweaks after initial deployment.


In [ ]:
print('Updating table annotations...')
vecdb.update_vector_table_annotation(
    name=TABLE,
    comment='Index/model demo table with OWNER annotation',
    annotations={'TITLE': 'string', 'CATEGORY': 'string', 'OWNER': 'string'}
)
vecdb.describe_vector_table(name=TABLE)

## 8. Cleanup
Reset the environment by tearing down the demo index and table so repeated runs start from a clean slate.


In [ ]:
print('Dropping index and table...')
try:
    vecdb.drop_index(table_name=TABLE, index_params={"index_type": "all"})
except Exception as exc:
    print('drop_index error:', exc)
vecdb.drop_vector_table(name=TABLE)
print('Cleanup complete')